<p>
  <b>AI Lab: Deep Learning for Computer Vision</b><br>
  <b><a href="https://www.wqu.edu/">WorldQuant University</a></b>
</p>

### Getting Ready

Before we can start this lesson, we need to import the required libraries.

In [1]:
import sys
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from IPython.display import Video
import torch
import torchvision
from torchvision import transforms
from torchvision.io import read_image
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import draw_bounding_boxes, make_grid

Next, print the version numbers of the primary software to improve reproducibility. 

In [2]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("CV2 version : ", cv2.__version__)
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)

Platform: linux
Python version: 3.11.0 (main, Nov 15 2022, 20:12:54) [GCC 10.2.1 20210110]
---
CV2 version :  4.10.0
torch version :  2.2.2+cu121
torchvision version :  0.17.2+cu121


 We are ready to start looking at the data. 🏎️💨

### Exploring Our Data

**Task 3.2.1:** Create a variable for the train directory using `pathlib` syntax.

In [4]:
dhaka_image_dir = Path("data_images")/ "train"

print("Data directory:", dhaka_image_dir)

Data directory: data_images/train


Let's examine some of the contents of the train directory. You'll see two types of files:

1. `.xml` files: These contain the annotations for the images.
2. `.jpg` files: These are the actual image files.

Each image typically has a corresponding XML file.

In [5]:
dhaka_files = list(dhaka_image_dir.iterdir())
dhaka_files[-5:]

[PosixPath('data_images/train/Dipto_442.xml'),
 PosixPath('data_images/train/Numan_(56).jpg'),
 PosixPath('data_images/train/Navid_323.xml'),
 PosixPath('data_images/train/Navid_97.jpg'),
 PosixPath('data_images/train/Navid_181.jpg')]

Even though we only see one type of image file, it turns out that the image files can have many different possible extensions. Let's count the file extensions by type and print the results.

In [6]:
file_extension_counts = Counter(Path(file).suffix for file in dhaka_files)

for extension, count in file_extension_counts.items():
    print(f"Files with extension {extension}: {count}")

Files with extension .xml: 3003
Files with extension .jpg: 2844
Files with extension .JPG: 143
Files with extension .png: 12
Files with extension .jpeg: 2
Files with extension .PNG: 2


### Separating images and bounding boxes data

Bounding boxes are rectangles around a detected object. All bounding box information is contained in the `.xml` files. The images have several different extensions.   It makes sense to separate the different file types into different folders. We'll want to put all `.xml` files in an annotations folder and the various image types in an images folder.

**Task 3.2.2:** Create variables for the images and annotations directories using `pathlib` syntax.

In [7]:
images_dir = dhaka_image_dir/"images"
annotations_dir = dhaka_image_dir/"annotations"

images_dir.mkdir(exist_ok=True)
annotations_dir.mkdir(exist_ok=True)

**Task 3.2.3:** Move files to the appropriate directory based on file extensions.

In [8]:
for file in dhaka_files:
    if file.suffix.lower() in (".jpg", ".jpeg", ".png"):
        target_dir = images_dir
    elif file.suffix.lower() == ".xml":
        target_dir = annotations_dir
    file.rename(target_dir / file.name)

Let's confirm that all the files where moved by making sure there is equal number of images and annotations. 

In [9]:
images_files = list(images_dir.iterdir())
annotations_files = list(annotations_dir.iterdir())

assert len(images_files) == len(annotations_files)

### Annotations

The annotations are the labels for the data. Each image has an annotation that contains the coordinates and type of object for each bounding box in a given image.

Let's look at the structure of the annotations by loading the first 25 lines of a file. The annotations are stored as XML which is a way to store structured documents. The `<annotation>` tag is the root element, containing all the information about this particular image annotation. The tags within store other information such as `<folder>`. The most important tag for the current project is the `<object>`. It describes an object detected in the image, this associated image contains a "bus". The tag `<bndbox>` is the bounding box information. There are the coordinates of a rectangle surrounding the bus in the image (in pixels): `<xmin>` is the left edge, `<ymin>` is the top edge, `<xmax>` is the right edge, and `<ymax>` is the bottom edge.

In [49]:
xml_filepath = annotations_dir / "10.xml"
!head -n 25 $xml_filepath

<annotation>
	<folder>Images</folder>
	<filename>79841485_748949428950493_32369669566365696_n.jpg</filename>
	<path>E:\Datasets\Dataset\Images\79841485_748949428950493_32369669566365696_n.jpg</path>
	<source>
		<database>Unknown</database>
	</source>
	<size>
		<width>749</width>
		<height>416</height>
		<depth>3</depth>
	</size>
	<segmented>0</segmented>
	<object>
		<name>car</name>
		<pose>Unspecified</pose>
		<truncated>1</truncated>
		<difficult>0</difficult>
		<bndbox>
			<xmin>120</xmin>
			<ymin>375</ymin>
			<xmax>189</xmax>
			<ymax>416</ymax>
		</bndbox>
	</object>


The ElementTree (ET) module in Python can parse an XML file. In XML, the root is the top-level element that contains all other elements. The `tag` attribute contains the name of the element. 

In [50]:
tree = ET.parse(xml_filepath)
root = tree.getroot()
print(root.tag)

annotation


The `find` method is used to locate the first occurrence of a sub-element with a given tag. Let's find the width and height of the image.

In [51]:
width = int(root.find("size").find("width").text)
height = int(root.find("size").find("height").text)
print(f"image width: {width}  image height: {height}")

image width: 749  image height: 416


The `findall` method finds all occurrences of a sub-element within a given tag. We can use that method to get all the relevant data for the bounding boxes.

**Task 3.2.4:** Find the labels and coordinates for all the bounding boxes.

In [56]:
bounding_boxes = []
labels = []
for obj in root.findall("object"):
    label = obj.find("name").text
    labels.append(label)
    bndbox = obj.find("bndbox")
    xmin = int(bndbox.find("xmin").text)
    ymin = int(bndbox.find("ymin").text)
    xmax = int(bndbox.find("xmax").text)
    ymax = int(bndbox.find("ymax").text)
    bounding_boxes.append([xmin, ymin, xmax, ymax])

for label, bounding_box in zip(labels, bounding_boxes):
    print(f"{label}: {bounding_box}")

car: [120, 375, 189, 416]
minivan: [212, 369, 266, 416]
car: [289, 397, 339, 416]
car: [390, 350, 444, 394]
van: [148, 325, 197, 377]
motorbike: [362, 373, 380, 404]
motorbike: [112, 369, 132, 396]
pickup: [213, 325, 258, 380]
car: [430, 333, 478, 371]
car: [78, 313, 156, 342]
car: [269, 359, 327, 406]
motorbike: [323, 362, 343, 385]
car: [332, 329, 375, 367]
car: [417, 315, 452, 345]
human hauler: [358, 304, 397, 345]
van: [234, 299, 269, 344]
three wheelers (CNG): [186, 307, 211, 343]
three wheelers (CNG): [292, 330, 322, 367]
minivan: [275, 311, 314, 354]
car: [319, 315, 358, 340]
bus: [219, 265, 258, 320]
car: [346, 295, 369, 317]
bus: [337, 248, 387, 288]
bus: [241, 245, 278, 274]
bus: [320, 243, 339, 272]
bus: [300, 233, 319, 257]
car: [300, 281, 326, 296]


### Bounding boxes in PyTorch

**Task 3.2.5:** Convert bounding boxes to PyTorch tensors.

In [57]:
bboxes_tensor = torch.tensor(bounding_boxes ,dtype=torch.float)
print(bboxes_tensor)

tensor([[120., 375., 189., 416.],
        [212., 369., 266., 416.],
        [289., 397., 339., 416.],
        [390., 350., 444., 394.],
        [148., 325., 197., 377.],
        [362., 373., 380., 404.],
        [112., 369., 132., 396.],
        [213., 325., 258., 380.],
        [430., 333., 478., 371.],
        [ 78., 313., 156., 342.],
        [269., 359., 327., 406.],
        [323., 362., 343., 385.],
        [332., 329., 375., 367.],
        [417., 315., 452., 345.],
        [358., 304., 397., 345.],
        [234., 299., 269., 344.],
        [186., 307., 211., 343.],
        [292., 330., 322., 367.],
        [275., 311., 314., 354.],
        [319., 315., 358., 340.],
        [219., 265., 258., 320.],
        [346., 295., 369., 317.],
        [337., 248., 387., 288.],
        [241., 245., 278., 274.],
        [320., 243., 339., 272.],
        [300., 233., 319., 257.],
        [300., 281., 326., 296.]])
